In [ ]:
# PS10-Team ATX
# ─────────────────────────────────────────────────────────────────────────────
import torch
import subprocess

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    raise RuntimeError("❌ No GPU found! Go to Settings → Accelerator → GPU T4 x1 and restart.")



PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM (GB): 15.6


In [ ]:

# ═══════════════════════════════════════════════════════════════════
# CELL 2 — Install dependencies
# ═══════════════════════════════════════════════════════════════════
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

pip("rasterio")
pip("tifffile")
pip("scikit-image")
pip("pytorch-fid")

import rasterio, tifffile, skimage
print("rasterio OK:", rasterio.__version__)
print("tifffile OK:", tifffile.__version__)
print("scikit-image OK:", skimage.__version__)



rasterio OK: 1.5.0
tifffile OK: 2026.4.11
scikit-image OK: 0.25.2


In [ ]:

# ═══════════════════════════════════════════════════════════════════
# CELL 3 — Upload & extract project files
# ═══════════════════════════════════════════════════════════════════
import os, shutil, sys

PROJECT_SRC = "/kaggle/input/datasets/mahatvaofficial/ps10-new/ps10_project"   # adjust if dataset name differs
WORK_DIR    = "/kaggle/working/ps10_project"

if os.path.exists(PROJECT_SRC):
    if os.path.exists(WORK_DIR):
        shutil.rmtree(WORK_DIR)
    shutil.copytree(PROJECT_SRC, WORK_DIR)
    print("Copied project to", WORK_DIR)
else:
    # Fallback: uploaded as zip directly in the notebook session
    import zipfile
    zip_path = "/kaggle/working/ps10_project.zip"
    with zipfile.ZipFile(zip_path) as z:
        z.extractall("/kaggle/working/")
    print("Extracted project zip")

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print("Working dir:", os.getcwd())
print("Files:", os.listdir("."))



Copied project to /kaggle/working/ps10_project
Working dir: /kaggle/working/ps10_project
Files: ['bonus_physics_informed.py', 'train_colorize.py', 'evaluate.py', 'train_sr.py', 'models', 'README-2.md', 'requirements.txt', 'inference.py', 'TEST_REPORT.md', 'tests', 'README_SUBMISSION.md', 'datasets']


In [ ]:
import os


if not os.path.exists("/kaggle/working/ISRO"):
    !git clone --depth=1 https://github.com/jugal-sac/IR-colorization-BAH2026.git /kaggle/working/ISRO
    print("✅ Cloned ISRO repo")
else:
    print("✅ ISRO repo already exists")


!cat /kaggle/working/ISRO/scripts/download_data.sh



Cloning into '/kaggle/working/ISRO'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 31 (delta 0), reused 30 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 2.57 MiB | 28.93 MiB/s, done.
✅ Cloned ISRO repo
#!/bin/bash
# Simple wrapper for the download script
echo "Starting data download..."
# Example product ID and bands. 
# Users should edit these or create a config file.
PRODUCT_ID="LC9123456789"
BANDS="SR_B2,SR_B3,SR_B4,ST_B10"
START_DATE="2023-01-01"
END_DATE="2023-12-31"
OUTPUT_PATH="./input/${PRODUCT_ID}_product"
EE_PROJECT_ID="your-google-earth-engine-project-id"

python scripts/download.py $PRODUCT_ID $BANDS $START_DATE $END_DATE $OUTPUT_PATH --ee_project_id $EE_PROJECT_ID


In [ ]:
import ee
ee.Authenticate()


In [46]:
EE_PROJECT_ID="datzards"

In [ ]:
import ee
PROJECT_ID = "datzards" 

ee.Initialize(project=PROJECT_ID)
print("✅ Successfully initialized Earth Engine!")


✅ Successfully initialized Earth Engine!


In [ ]:
import sys
import os
import ee
import geemap

%cd /kaggle/working/ISRO
os.makedirs("input/demo_product", exist_ok=True)

start_date = '2023-01-01'
end_date = '2023-12-31'

print("Connecting to Earth Engine...")
ee.Initialize(project="datzards")

roi = ee.Geometry.Rectangle([76.5, 27.5, 78.0, 29.0]) 


collection = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
              .filterBounds(roi) # Filter collection to only keep assets matching our ROI
              .filterDate(start_date, end_date)
              .sort('CLOUD_COVER') 
              .first())

if collection is None:
    print("🚨 Could not find valid imagery matching this localized region.")
else:
    # Clip our image tightly to the bounding box to keep the download size under 50MB
    clipped_image = collection.clip(roi)
    print("Selected Image ID:", collection.get('LANDSAT_PRODUCT_ID').getInfo())
    
    target_bands = {
        'SR_B2': 'input/demo_product/demo_B2.TIF',
        'SR_B3': 'input/demo_product/demo_B3.TIF',
        'SR_B4': 'input/demo_product/demo_B4.TIF',
        'ST_B10': 'input/demo_product/demo_B10.TIF'
    }

    # 3. Export each channel band with the ROI restriction applied
    for band_name, output_filename in target_bands.items():
        print(f"Downloading {band_name}...")
        single_band_image = clipped_image.select(band_name)
        
        geemap.ee_export_image(
            single_band_image, 
            filename=output_filename, 
            scale=30, 
            region=roi # Explicitly pass the small bounding box here
        )

print("Everything completed! Checking output folder contents:")
!ls -lh input/demo_product/


/kaggle/working/ISRO
Connecting to Earth Engine...
Selected Image ID: LC09_L2SP_146041_20231008_20231009_02_T1
Generating URL ...
An error occurred while downloading.
Total request size (84695922 bytes) must be less than or equal to 50331648 bytes.
Generating URL ...
An error occurred while downloading.
Total request size (84695922 bytes) must be less than or equal to 50331648 bytes.
Generating URL ...
An error occurred while downloading.
Total request size (84695922 bytes) must be less than or equal to 50331648 bytes.
Generating URL ...
An error occurred while downloading.
Total request size (84695922 bytes) must be less than or equal to 50331648 bytes.
Everything completed! Checking output folder contents:
total 5.1M
-rw-r--r-- 1 root root 1.3M Jul  1 10:23 demo_B10.tif
-rw-r--r-- 1 root root 1.3M Jul  1 10:23 demo_B2.tif
-rw-r--r-- 1 root root 1.3M Jul  1 10:23 demo_B3.tif
-rw-r--r-- 1 root root 1.3M Jul  1 10:23 demo_B4.tif


In [66]:
!grep -n "glob\|\.TIF\|\.tif\|B10\|B2\|B3\|B4\|input" /kaggle/working/ISRO/driver.py | head -40


33:    input_root = os.path.join(base_dir, 'input')
45:    if not os.path.isdir(input_root):
46:        logger.error(f"Input root directory {input_root} not found.")
49:    product_folders = [e for e in os.listdir(input_root) if os.path.isdir(os.path.join(input_root, e))]
52:        input_dir = os.path.join(input_root, product_id)
55:        band2_path = find_file(input_dir, '_B2')
56:        band3_path = find_file(input_dir, '_B3')
57:        band4_path = find_file(input_dir, '_B4')
58:        band10_path = find_file(input_dir, '_B10')
68:            rgb_output_path = os.path.join(output_rgb_dir, f'{file_prefix}_rgb_30m.tif')
72:            downscaled_rgb_100m = os.path.join(output_downscale_dir, f'{file_prefix}_rgb_100m.tif')
76:            downscaled_tir_100m = os.path.join(output_downscale_dir, f'{file_prefix}_tir_100m.tif')
80:            downscaled_tir_200m = os.path.join(output_downscale_dir, f'{file_prefix}_tir_200m.tif')
84:            run_script('create_patches.py', logger, '

In [67]:
import subprocess, os

os.chdir("/kaggle/working/ISRO")

result = subprocess.run(
    ["python", "driver.py"],
    capture_output=True, text=True
)
print(result.stdout[-3000:])   # last 3000 chars of output
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])


In [ ]:
import glob, numpy as np

patches = glob.glob("output/patches/**/*.npy", recursive=True)
print(f"Total .npy patches: {len(patches)}")

# Show a sample of filenames 
for f in sorted(patches)[:10]:
    arr = np.load(f)
    print(f"  {os.path.basename(f):45s}  shape={arr.shape}  dtype={arr.dtype}")


Total .npy patches: 3
  rgb_100m_512.npy                               shape=(3, 512, 512)  dtype=uint16
  tir_100m_512.npy                               shape=(1, 512, 512)  dtype=uint16
  tir_200m.npy                                   shape=(1, 256, 256)  dtype=uint16


In [92]:
import os, glob

demo_dir = "/kaggle/working/ISRO/input/demo_product"
files = glob.glob(f"{demo_dir}/**/*.TIF", recursive=True) + \
        glob.glob(f"{demo_dir}/**/*.tif", recursive=True)

print(f"Found {len(files)} TIF files:")
for f in sorted(files):
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {os.path.basename(f):40s}  {size_mb:.1f} MB")

# Check all 4 required bands are present
required = ["_B2.TIF", "_B3.TIF", "_B4.TIF", "_B10.TIF"]
for band in required:
    found = any(band.upper() in f.upper() for f in files)
    print(f"  {'✅' if found else '❌'}  {band}")


Found 4 TIF files:
  demo_B10.tif                              1.3 MB
  demo_B2.tif                               1.3 MB
  demo_B3.tif                               1.3 MB
  demo_B4.tif                               1.4 MB
  ✅  _B2.TIF
  ✅  _B3.TIF
  ✅  _B4.TIF
  ✅  _B10.TIF


In [ ]:
import ee, geemap, os
%cd /kaggle/working/ISRO

ee.Initialize(project="datzards")

sub_rois = {
    "tile_SW": ee.Geometry.Rectangle([76.5,  27.5,  77.25, 28.25]),
    "tile_SE": ee.Geometry.Rectangle([77.25, 27.5,  78.0,  28.25]),
    "tile_NW": ee.Geometry.Rectangle([76.5,  28.25, 77.25, 29.0]),
    "tile_NE": ee.Geometry.Rectangle([77.25, 28.25, 78.0,  29.0]),
}

bands = {"SR_B2": "demo_B2.TIF", "SR_B3": "demo_B3.TIF",
         "SR_B4": "demo_B4.TIF", "ST_B10": "demo_B10.TIF"}

for tile_name, roi in sub_rois.items():
    out_dir = f"input/{tile_name}"
    os.makedirs(out_dir, exist_ok=True)
    
    print(f"\n── Downloading {tile_name} ──")
    img = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
           .filterBounds(roi)
           .filterDate('2023-01-01', '2023-12-31')
           .sort('CLOUD_COVER')
           .first()
           .clip(roi))
    
    print("  Scene:", img.get('LANDSAT_PRODUCT_ID').getInfo())
    
    for band, fname in bands.items():
        print(f"  Downloading {band}...")
        geemap.ee_export_image(
            img.select(band),
            filename=f"{out_dir}/{fname}",
            scale=30,
            region=roi
        )

print("\n✅ All tiles downloaded. Checking:")
!ls -lh input/tile_SW/ input/tile_SE/ input/tile_NW/ input/tile_NE/


/kaggle/working/ISRO

── Downloading tile_SW ──
  Scene: LC09_L2SP_146041_20231008_20231009_02_T1
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/ISRO/input/tile_SW/demo_B2.TIF
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/ISRO/input/tile_SW/demo_B3.TIF
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/ISRO/input/tile_SW/demo_B4.TIF
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/ISRO/input/tile_SW/demo_B10.TIF

── Downloading tile_SE ──
  Scene: LC09_L2SP_146041_20231008_20231009_02_T1
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/ISRO/input/tile_SE/demo_B2.TIF
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/ISRO/input/tile_SE/demo_B3.TIF
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/ISRO/input/tile_SE/demo_B4.TIF
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/ISRO/input/tile_SE/demo_B10.TIF

── Downloading

In [91]:
import subprocess, os

os.chdir("/kaggle/working/ISRO")

tiles = ["tile_SW", "tile_SE", "tile_NW", "tile_NE"]

for tile in tiles:
    print(f"\n── Processing {tile} ──")
    # driver.py processes everything in input/ automatically
    # Each tile folder is a separate "product" it will pick up
    pass

# Run driver.py once — it processes ALL subfolders in input/ in one go
result = subprocess.run(["python", "driver.py"], capture_output=True, text=True)
print(result.stdout[-4000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])



── Processing tile_SW ──

── Processing tile_SE ──

── Processing tile_NW ──

── Processing tile_NE ──



In [78]:
# ── Augmented dataset wrappers ────────────────────────────────────────
import torch, random
import numpy as np
from torch.utils.data import Dataset

class AugmentedSRDataset(Dataset):
    """Wraps SRPatchDataset with 8× augmentation (4 rotations × 2 flips)."""
    def __init__(self, base_dataset):
        self.base = base_dataset
        self.n_aug = 8  # 4 rotations × 2 horizontal flips

    def __len__(self):
        return len(self.base) * self.n_aug

    def __getitem__(self, idx):
        low, high = self.base[idx % len(self.base)]
        aug_id = idx // len(self.base)
        k   = aug_id % 4          # rotation: 0/90/180/270
        flip = aug_id // 4         # 0=no flip, 1=hflip
        low  = torch.rot90(low,  k, dims=[-2, -1])
        high = torch.rot90(high, k, dims=[-2, -1])
        if flip:
            low  = torch.flip(low,  dims=[-1])
            high = torch.flip(high, dims=[-1])
        return low, high


class AugmentedColorDataset(Dataset):
    """Wraps ColorizationPatchDataset with 8× augmentation."""
    def __init__(self, base_dataset):
        self.base = base_dataset
        self.n_aug = 8

    def __len__(self):
        return len(self.base) * self.n_aug

    def __getitem__(self, idx):
        tir, rgb = self.base[idx % len(self.base)]
        aug_id = idx // len(self.base)
        k   = aug_id % 4
        flip = aug_id // 4
        tir = torch.rot90(tir, k, dims=[-2, -1])
        rgb = torch.rot90(rgb, k, dims=[-2, -1])
        if flip:
            tir = torch.flip(tir, dims=[-1])
            rgb = torch.flip(rgb, dims=[-1])
        return tir, rgb

# ── Verify patch count after running driver.py ────────────────────────
import glob, numpy as np, os, sys
sys.path.insert(0, "/kaggle/working/ps10_project/datasets")
from patch_dataset import SRPatchDataset, ColorizationPatchDataset

patches = glob.glob("/kaggle/working/ISRO/output/patches/**/*.npy", recursive=True)
print(f"Total patches from driver.py: {len(patches)}")
for f in sorted(patches):
    arr = np.load(f)
    print(f"  {os.path.basename(f):45s}  {arr.shape}  {arr.dtype}")


Total patches from driver.py: 6
  rgb_100m_512.npy                               (3, 512, 512)  uint16
  tir_100m_512.npy                               (1, 512, 512)  uint16
  tir_200m.npy                                   (1, 256, 256)  uint16
  rgb_100m_512.npy                               (3, 512, 512)  uint16
  tir_100m_512.npy                               (1, 512, 512)  uint16
  tir_200m.npy                                   (1, 256, 256)  uint16


In [ ]:

import glob as _glob, os as _os, re as _re
from collections import defaultdict

def _fixed_index(self, low_glob, high_glob):
    lows = sorted(set(
        _glob.glob(_os.path.join(self.patches_dir, low_glob)) +
        _glob.glob(_os.path.join(self.patches_dir, "**", low_glob), recursive=True)
    ))
    highs = sorted(set(
        _glob.glob(_os.path.join(self.patches_dir, high_glob)) +
        _glob.glob(_os.path.join(self.patches_dir, "**", high_glob), recursive=True)
    ))

    def digit_id(p):
        stem = _os.path.splitext(_os.path.basename(p))[0]
        m = _re.search(r"(\d+)$", stem)
        return m.group(1) if m else ""

    low_by_dir  = defaultdict(list)
    high_by_dir = defaultdict(list)
    for f in lows:
        low_by_dir[_os.path.dirname(_os.path.abspath(f))].append(f)
    for f in highs:
        high_by_dir[_os.path.dirname(_os.path.abspath(f))].append(f)

    pairs = []
    for d in sorted(set(low_by_dir) & set(high_by_dir)):
        l, h = low_by_dir[d], high_by_dir[d]
        if len(l) == 1 and len(h) == 1:
            # One of each in this dir = driver.py per-tile output, pair directly
            pairs.append((l[0], h[0]))
        else:
            # Multiple files = flat numbered layout, pair by digit ID
            lb = {digit_id(f): f for f in l}
            hb = {digit_id(f): f for f in h}
            for k in sorted(set(lb) & set(hb)):
                pairs.append((lb[k], hb[k]))
    return pairs

# Apply the fix
from patch_dataset import _BasePatchIndex
_BasePatchIndex._index = _fixed_index
print("✅ patch_dataset._index patched")


✅ patch_dataset._index patched


In [ ]:
import sys, glob, os, numpy as np
sys.path.insert(0, "/kaggle/working/ps10_project")


from patch_dataset import SRPatchDataset, ColorizationPatchDataset

PATCHES = "/kaggle/working/ISRO/output/patches"

try:
    sr = SRPatchDataset(PATCHES,
                        expected_low_shape=(1, 256, 256),   # ← 3D shape
                        expected_high_shape=(1, 512, 512))  # ← 3D shape
    print(f"✅ SR pairs: {len(sr)}")
    for i, (a, b) in enumerate(sr.pairs):
        print(f"   [{i}] {os.path.basename(a)} → {os.path.basename(b)}")
    low, high = sr[0]
    print(f"   Tensors: low={low.shape}  high={high.shape}")
except Exception as e:
    print(f"❌ SR: {e}")

try:
    col = ColorizationPatchDataset(PATCHES,
                                   expected_tir_shape=(1, 512, 512))  # ← 3D shape
    print(f"\n✅ Color pairs: {len(col)}")
    for i, (a, b) in enumerate(col.pairs):
        print(f"   [{i}] {os.path.basename(a)} → {os.path.basename(b)}")
    tir, rgb = col[0]
    print(f"   Tensors: tir={tir.shape}  rgb={rgb.shape}")
except Exception as e:
    print(f"❌ Color: {e}")


✅ SR pairs: 2
   [0] tir_200m.npy → tir_100m_512.npy
   [1] tir_200m.npy → tir_100m_512.npy
   Tensors: low=torch.Size([1, 256, 256])  high=torch.Size([1, 512, 512])

✅ Color pairs: 2
   [0] tir_100m_512.npy → rgb_100m_512.npy
   [1] tir_100m_512.npy → rgb_100m_512.npy
   Tensors: tir=torch.Size([1, 512, 512])  rgb=torch.Size([3, 512, 512])


In [86]:
%%writefile /kaggle/working/ps10_project/datasets/patch_dataset.py
"""
Dataset classes reading the .npy patches produced by the original repo's
driver.py (see output/patches/ per the README).

IMPORTANT: the README describes the *shapes and pairing* precisely:
    SR pair:            256x256 (200m TIR)  ->  512x512 (100m TIR)
    Colorization pair:  256x256 (100m TIR)   ->  256x256 (100m RGB)
but does not give exact filenames -- driver.py wasn't in the files I had
access to. This loader assumes a reasonably conventional naming scheme
(one .npy per patch, per band/resolution, sharing a common patch ID) and
is written so the glob patterns in `_index()` are the ONE place you need
to edit once you've actually run driver.py and can see real filenames in
output/patches/.

ASSUMPTION THAT NEEDS VERIFYING: SR-target patches and colorization-input
patches are BOTH "100m TIR" but at DIFFERENT patch sizes (512x512 vs
256x256) per the README. If driver.py saves them into the same flat
directory with similar names, a naive glob can't tell them apart by name
alone. This loader therefore (a) expects SR patches and colorization
patches to live in separate subdirectories -- pass --sr_patches_dir and
--colorize_patches_dir separately once you've run driver.py and can see
its real output layout -- and (b) validates array shape against the
expected size on load and raises immediately if it doesn't match, so a
mis-pointed directory fails loudly instead of silently training on the
wrong pairs.

Run `python datasets/patch_dataset.py <sr_dir> <colorize_dir>` after
generating real patches to sanity-check indexing before training.
"""
import glob
import os
import re
from typing import List, Tuple

import numpy as np

try:
    import torch
    from torch.utils.data import Dataset
except ImportError:  # allows the indexing/shape logic to be smoke-tested without torch
    torch = None
    Dataset = object


def _normalize(arr: np.ndarray) -> np.ndarray:
    """Min-max normalize a patch to [-1, 1]. Radiometric range is scene-
    dependent, so per-patch normalization is used (documented tradeoff:
    this discards absolute radiance information the physics-informed loss
    could otherwise use -- see bonus_physics_informed.py for a variant
    that keeps raw radiance alongside the normalized tensor)."""
    arr = arr.astype(np.float64)  # float64 avoids float32-precision loss in the
    # near-constant-patch epsilon guard below (a small additive epsilon can be
    # silently swallowed by float32 rounding when lo/hi are large, e.g. raw
    # digital-number radiance values in the thousands)
    lo, hi = np.percentile(arr, 1), np.percentile(arr, 99)
    span = hi - lo
    if span < 1e-6:
        # relative epsilon: safe even when lo/hi are large radiance values
        eps = max(1e-6, abs(lo) * 1e-6, 1.0)
        hi = lo + eps
    arr = np.clip(arr, lo, hi)
    arr = 2 * (arr - lo) / (hi - lo) - 1
    return arr.astype(np.float32)


class _BasePatchIndex:
    """Shared file-discovery logic. EDIT THE GLOB PATTERNS to match your
    actual driver.py output once you've run it."""

    def __init__(self, patches_dir: str):
        self.patches_dir = patches_dir

    def _index(self, low_glob: str, high_glob: str) -> List[Tuple[str, str]]:
        # Recursive search: finds files in both the root dir and any
        # subdirectories (e.g. output/patches/tile_SW/, tile_NW/, ...).
        low_files = sorted(set(
            glob.glob(os.path.join(self.patches_dir, low_glob)) +
            glob.glob(os.path.join(self.patches_dir, "**", low_glob), recursive=True)
        ))
        high_files = sorted(set(
            glob.glob(os.path.join(self.patches_dir, high_glob)) +
            glob.glob(os.path.join(self.patches_dir, "**", high_glob), recursive=True)
        ))

        def digit_id(path: str) -> str:
            """Trailing digit run of the filename stem, or '' if none exists."""
            stem = os.path.splitext(os.path.basename(path))[0]
            m = re.search(r"(\d+)$", stem)
            return m.group(1) if m else ""

        # Group by parent directory: driver.py puts each tile's patch set
        # into one subdirectory, so same-dir files are spatially matched.
        from collections import defaultdict
        low_by_dir:  dict = defaultdict(list)
        high_by_dir: dict = defaultdict(list)
        for f in low_files:
            low_by_dir[os.path.dirname(os.path.abspath(f))].append(f)
        for f in high_files:
            high_by_dir[os.path.dirname(os.path.abspath(f))].append(f)

        pairs: List[Tuple[str, str]] = []
        for d in sorted(set(low_by_dir) & set(high_by_dir)):
            lows  = low_by_dir[d]
            highs = high_by_dir[d]

            if len(lows) == 1 and len(highs) == 1:
                # Exactly one of each in this directory (driver.py per-tile
                # output: tir_200m.npy + tir_100m_512.npy). Pair directly
                # regardless of digit suffix — they're the only match possible.
                pairs.append((lows[0], highs[0]))
            else:
                # Multiple files per directory (flat numbered layout e.g.
                # tir_200m_0000.npy, tir_200m_0001.npy, ...).
                # Match by trailing digit run as original logic.
                low_by_id  = {digit_id(f): f for f in lows}
                high_by_id = {digit_id(f): f for f in highs}
                common = sorted(set(low_by_id) & set(high_by_id),
                                key=lambda x: (len(x), x))
                pairs.extend([(low_by_id[k], high_by_id[k]) for k in common])

        return pairs


class SRPatchDataset(_BasePatchIndex, Dataset):
    """
    Yields (tir_200m, tir_100m) pairs, shapes (1,256,256) and (1,512,512),
    both normalized to [-1, 1].

    Actual driver.py filenames: tir_200m.npy (256x256) + tir_100m_512.npy (512x512).
    The glob patterns below are written to match this naming exactly.
    """

    def __init__(self, patches_dir: str,
                 low_glob: str = "*tir*200m*.npy",
                 high_glob: str = "*tir*100m*.npy",
                 expected_low_shape=(256, 256), expected_high_shape=(512, 512)):
        _BasePatchIndex.__init__(self, patches_dir)
        self.pairs = self._index(low_glob, high_glob)
        if len(self.pairs) == 0:
            raise RuntimeError(
                f"No SR pairs found in {patches_dir} with patterns "
                f"'{low_glob}' / '{high_glob}'. Check driver.py's actual "
                f"output filenames and adjust the glob patterns."
            )
        # Fail loudly if this directory holds the wrong patch type.
        # Use [-2:] to compare only spatial dims (H, W) — ignores the
        # channel dim so (1,256,256) and (256,256) both pass for (256,256).
        low0  = np.load(self.pairs[0][0]).shape
        high0 = np.load(self.pairs[0][1]).shape
        if tuple(low0[-2:]) != expected_low_shape or tuple(high0[-2:]) != expected_high_shape:
            raise RuntimeError(
                f"SRPatchDataset: expected spatial shapes {expected_low_shape} -> "
                f"{expected_high_shape} but found {low0} -> {high0} in "
                f"{patches_dir}. Are sr_patches_dir and "
                f"colorize_patches_dir pointed at the right folders?"
            )

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        low_path, high_path = self.pairs[idx]
        low = _normalize(np.load(low_path))
        high = _normalize(np.load(high_path))
        low_t = torch.from_numpy(low).unsqueeze(0) if low.ndim == 2 else torch.from_numpy(low)
        high_t = torch.from_numpy(high).unsqueeze(0) if high.ndim == 2 else torch.from_numpy(high)
        return low_t, high_t


class ColorizationPatchDataset(_BasePatchIndex, Dataset):
    """
    Yields (tir_100m, rgb_100m) pairs.

    Actual driver.py output: tir_100m_512.npy (1,512,512) and
    rgb_100m_512.npy (3,512,512). Despite the README describing 256x256
    colorization patches, the real driver produces 512x512 for both TIR
    and RGB at 100m. The colorization model (UNetGenerator) is trained at
    this native 512x512 size accordingly.
    """

    def __init__(self, patches_dir: str,
                 tir_glob: str = "*tir*100m*.npy",
                 rgb_glob: str = "*rgb*100m*.npy",
                 expected_tir_shape=(512, 512)):
        _BasePatchIndex.__init__(self, patches_dir)
        self.pairs = self._index(tir_glob, rgb_glob)
        if len(self.pairs) == 0:
            raise RuntimeError(
                f"No colorization pairs found in {patches_dir} with patterns "
                f"'{tir_glob}' / '{rgb_glob}'. Check driver.py's actual "
                f"output filenames and adjust the glob patterns."
            )
        tir0 = np.load(self.pairs[0][0]).shape
        # Compare only spatial dims [-2:] so (1,512,512) passes for (512,512).
        tir_spatial = tuple(tir0[-2:])
        valid_shapes = {expected_tir_shape, (256, 256), (512, 512)}
        if tir_spatial not in valid_shapes:
            raise RuntimeError(
                f"ColorizationPatchDataset: unexpected TIR spatial shape {tir_spatial} "
                f"(full shape {tir0}) in {patches_dir}. "
                f"Expected one of {valid_shapes}. "
                f"Check --colorize_patches_dir is pointed at the correct folder."
            )
        self.tir_hw = tir_spatial  # store actual HxW for model config

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        tir_path, rgb_path = self.pairs[idx]
        tir = _normalize(np.load(tir_path))
        rgb = np.load(rgb_path).astype(np.float32)
        # rgb patches are expected as (H, W, 3) from driver.py; normalize per-channel
        if rgb.ndim == 3 and rgb.shape[-1] == 3:
            rgb = np.stack([_normalize(rgb[..., c]) for c in range(3)], axis=0)  # -> (3,H,W)
        elif rgb.ndim == 3 and rgb.shape[0] == 3:
            rgb = np.stack([_normalize(rgb[c]) for c in range(3)], axis=0)
        else:
            raise ValueError(f"Unexpected RGB patch shape {rgb.shape} in {rgb_path}")

        tir_t = torch.from_numpy(tir).unsqueeze(0) if tir.ndim == 2 else torch.from_numpy(tir)
        rgb_t = torch.from_numpy(rgb)
        return tir_t, rgb_t


if __name__ == "__main__":
    import sys
    sr_dir = sys.argv[1] if len(sys.argv) > 1 else "output/patches/sr"
    color_dir = sys.argv[2] if len(sys.argv) > 2 else "output/patches/colorize"
    try:
        ds = SRPatchDataset(sr_dir)
        print(f"SR dataset: {len(ds)} pairs found in {sr_dir}")
    except RuntimeError as e:
        print(f"SR dataset: {e}")
    try:
        ds = ColorizationPatchDataset(color_dir)
        print(f"Colorization dataset: {len(ds)} pairs found in {color_dir}")
    except RuntimeError as e:
        print(f"Colorization dataset: {e}")


Overwriting /kaggle/working/ps10_project/datasets/patch_dataset.py


In [58]:
!cd /kaggle/working/ISRO   # the cloned ISRO repo
!python driver.py


2026-07-01 10:25:35,540 - /kaggle/working/ISRO/output - INFO - Processing product: demo_product
2026-07-01 10:25:35,540 - /kaggle/working/ISRO/output - INFO - Running: python /kaggle/working/ISRO/scripts/merge_rgb.py /kaggle/working/ISRO/input/demo_product/demo_B4.tif /kaggle/working/ISRO/input/demo_product/demo_B3.tif /kaggle/working/ISRO/input/demo_product/demo_B2.tif /kaggle/working/ISRO/output/rgb_images/demo_product_rgb_30m.tif
2026-07-01 10:25:36,387 - /kaggle/working/ISRO/output - WARNING - STDERR from merge_rgb.py:
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")
<tifffile.TiffPage 0 @8> parsing GDAL_NODATA tag raised ValueError("invalid literal for int() with base 10: '0.0'")

2026-07-01 10:25:36,388 - /kaggle/working/ISRO/output - INFO - Running: python /kaggle/working/ISRO/scripts/downscale.py /

In [95]:
import sys, os, glob, re
import numpy as np
import torch
from torch.utils.data import Dataset
sys.path.insert(0, "/kaggle/working/ps10_project")

# ── 1. Monkey-Patch the old code on Kaggle ─────────────────────────────────
from patch_dataset import _BasePatchIndex, SRPatchDataset, ColorizationPatchDataset

def _fixed_index(self, low_glob, high_glob):
    lows = sorted(set(glob.glob(os.path.join(self.patches_dir, low_glob)) + glob.glob(os.path.join(self.patches_dir, "**", low_glob), recursive=True)))
    highs = sorted(set(glob.glob(os.path.join(self.patches_dir, high_glob)) + glob.glob(os.path.join(self.patches_dir, "**", high_glob), recursive=True)))
    def digit_id(p):
        m = re.search(r"(\d+)$", os.path.splitext(os.path.basename(p))[0])
        return m.group(1) if m else ""
    from collections import defaultdict
    lb_dir, hb_dir = defaultdict(list), defaultdict(list)
    for f in lows: lb_dir[os.path.dirname(os.path.abspath(f))].append(f)
    for f in highs: hb_dir[os.path.dirname(os.path.abspath(f))].append(f)
    pairs = []
    for d in sorted(set(lb_dir) & set(hb_dir)):
        l, h = lb_dir[d], hb_dir[d]
        if len(l) == 1 and len(h) == 1: pairs.append((l[0], h[0]))
        else:
            lb, hb = {digit_id(f): f for f in l}, {digit_id(f): f for f in h}
            for k in sorted(set(lb) & set(hb)): pairs.append((lb[k], hb[k]))
    return pairs
_BasePatchIndex._index = _fixed_index

# ── 2. Create the Datasets (Passing the 3D shapes to avoid the shape error) ──
PATCHES = "/kaggle/working/ISRO/output/patches"

print("── Checking Base Datasets ──")
base_sr = SRPatchDataset(PATCHES, expected_low_shape=(1, 256, 256), expected_high_shape=(1, 512, 512))
print(f"✅ SR pairs found: {len(base_sr)}")

base_color = ColorizationPatchDataset(PATCHES, expected_tir_shape=(1, 512, 512))
print(f"✅ Color pairs found: {len(base_color)}")

# ── 3. Create Augmented Datasets (8x multiplier) ───────────────────────────
class AugmentedSRDataset(Dataset):
    def __init__(self, base): self.base = base; self.n = 8
    def __len__(self): return len(self.base) * self.n
    def __getitem__(self, idx):
        low, high = self.base[idx % len(self.base)]
        k = (idx // len(self.base)) % 4
        f = idx // (len(self.base) * 4)
        low  = torch.rot90(low,  k, [-2,-1]); high = torch.rot90(high, k, [-2,-1])
        if f: low = torch.flip(low,[-1]); high = torch.flip(high,[-1])
        return low, high

class AugmentedColorDataset(Dataset):
    def __init__(self, base): self.base = base; self.n = 8
    def __len__(self): return len(self.base) * self.n
    def __getitem__(self, idx):
        tir, rgb = self.base[idx % len(self.base)]
        k = (idx // len(self.base)) % 4
        f = idx // (len(self.base) * 4)
        tir = torch.rot90(tir, k, [-2,-1]); rgb = torch.rot90(rgb, k, [-2,-1])
        if f: tir = torch.flip(tir,[-1]); rgb = torch.flip(rgb,[-1])
        return tir, rgb

aug_sr = AugmentedSRDataset(base_sr)
aug_color = AugmentedColorDataset(base_color)

print("\n── Final Training Sets ──")
print(f"🚀 SR Training Size:    {len(aug_sr)} patches")
print(f"🚀 Color Training Size: {len(aug_color)} patches")


── Checking Base Datasets ──
✅ SR pairs found: 2
✅ Color pairs found: 2

── Final Training Sets ──
🚀 SR Training Size:    16 patches
🚀 Color Training Size: 16 patches


In [107]:
import os
import torch
from torch.utils.data import Dataset
import numpy as np
from ps10_project.datasets.patch_dataset import SRPatchDataset, ColorizationPatchDataset, _normalize

PATCH_DIR = "/kaggle/working/ISRO/output/patches"

# ── 1. Bulletproof Dataset Overrides ──────────────────────────────────────────
class BulletproofSRDataset(Dataset):
    def __init__(self, patches_dir):
        self.pairs = []
        for root, dirs, files in os.walk(patches_dir):
            if "tir_200m.npy" in files and "tir_100m_512.npy" in files:
                self.pairs.append((os.path.join(root, "tir_200m.npy"), 
                                   os.path.join(root, "tir_100m_512.npy")))
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        low = _normalize(np.load(self.pairs[idx][0]))
        high = _normalize(np.load(self.pairs[idx][1]))
        return torch.from_numpy(low).unsqueeze(0), torch.from_numpy(high).unsqueeze(0)

class BulletproofColorDataset(Dataset):
    def __init__(self, patches_dir):
        self.pairs = []
        for root, dirs, files in os.walk(patches_dir):
            if "tir_100m_512.npy" in files and "rgb_100m_512.npy" in files:
                self.pairs.append((os.path.join(root, "tir_100m_512.npy"), 
                                   os.path.join(root, "rgb_100m_512.npy")))
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        tir = _normalize(np.load(self.pairs[idx][0]))
        rgb = np.load(self.pairs[idx][1]).astype(np.float32)
        rgb = np.clip(rgb, 0, 255)
        rgb = 2 * (rgb / 255.0) - 1
        return torch.from_numpy(tir).unsqueeze(0), torch.from_numpy(rgb).permute(2, 0, 1)

base_sr = BulletproofSRDataset(PATCH_DIR)
print(f"✅ SR pairs found: {len(base_sr)}")

base_color = BulletproofColorDataset(PATCH_DIR)
print(f"✅ Color pairs found: {len(base_color)}")

# ── 2. Create Augmented Datasets (8x multiplier) ───────────────────────────
class AugmentedDataset(Dataset):
    def __init__(self, base): self.base = base; self.n = 8
    def __len__(self): return len(self.base) * self.n
    def __getitem__(self, idx):
        a, b = self.base[idx % len(self.base)]
        k, f = (idx // len(self.base)) % 4, idx // (len(self.base) * 4)
        a, b = torch.rot90(a, k, [-2,-1]), torch.rot90(b, k, [-2,-1])
        if f: a, b = torch.flip(a, [-1]), torch.flip(b, [-1])
        return a, b

aug_sr = AugmentedDataset(base_sr)
aug_color = AugmentedDataset(base_color)

print("\n── Final Training Sets ──")
print(f"🚀 SR Training Size:    {len(aug_sr)} patches")
print(f"🚀 Color Training Size: {len(aug_color)} patches")


✅ SR pairs found: 2
✅ Color pairs found: 2

── Final Training Sets ──
🚀 SR Training Size:    16 patches
🚀 Color Training Size: 16 patches


In [113]:
import os
import torch
from torch.utils.data import Dataset
import numpy as np
from ps10_project.datasets.patch_dataset import _normalize

PATCH_DIR = "/kaggle/working/ISRO/output/patches"

# ── 1. Bulletproof Dataset Overrides ──────────────────────────────────────────
class BulletproofSRDataset(Dataset):
    def __init__(self, patches_dir):
        self.pairs = []
        for root, dirs, files in os.walk(patches_dir):
            if "tir_200m.npy" in files and "tir_100m_512.npy" in files:
                self.pairs.append((os.path.join(root, "tir_200m.npy"), 
                                   os.path.join(root, "tir_100m_512.npy")))
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        low = _normalize(np.load(self.pairs[idx][0]))
        high = _normalize(np.load(self.pairs[idx][1]))
        
        low_t = torch.from_numpy(low)
        high_t = torch.from_numpy(high)
        # Only unsqueeze if missing the channel dimension (ndim==2)
        if low_t.ndim == 2: low_t = low_t.unsqueeze(0)
        if high_t.ndim == 2: high_t = high_t.unsqueeze(0)
            
        return low_t, high_t

class BulletproofColorDataset(Dataset):
    def __init__(self, patches_dir):
        self.pairs = []
        for root, dirs, files in os.walk(patches_dir):
            if "tir_100m_512.npy" in files and "rgb_100m_512.npy" in files:
                self.pairs.append((os.path.join(root, "tir_100m_512.npy"), 
                                   os.path.join(root, "rgb_100m_512.npy")))
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        tir = _normalize(np.load(self.pairs[idx][0]))
        rgb = np.load(self.pairs[idx][1]).astype(np.float32)
        rgb = np.clip(rgb, 0, 255)
        rgb = 2 * (rgb / 255.0) - 1
        
        tir_t = torch.from_numpy(tir)
        if tir_t.ndim == 2: tir_t = tir_t.unsqueeze(0)
            
        # rgb is already (3, 512, 512) from driver.py, no permute needed
        rgb_t = torch.from_numpy(rgb)
        return tir_t, rgb_t

base_sr = BulletproofSRDataset(PATCH_DIR)
print(f"✅ SR pairs found: {len(base_sr)}")

base_color = BulletproofColorDataset(PATCH_DIR)
print(f"✅ Color pairs found: {len(base_color)}")

# ── 2. Create Augmented Datasets (8x multiplier) ───────────────────────────
class AugmentedDataset(Dataset):
    def __init__(self, base): self.base = base; self.n = 8
    def __len__(self): return len(self.base) * self.n
    def __getitem__(self, idx):
        a, b = self.base[idx % len(self.base)]
        k, f = (idx // len(self.base)) % 4, idx // (len(self.base) * 4)
        a, b = torch.rot90(a, k, [-2,-1]), torch.rot90(b, k, [-2,-1])
        if f: a, b = torch.flip(a, [-1]), torch.flip(b, [-1])
        return a, b

aug_sr = AugmentedDataset(base_sr)
aug_color = AugmentedDataset(base_color)

print("\n── Final Training Sets ──")
print(f"🚀 SR Training Size:    {len(aug_sr)} patches")
print(f"🚀 Color Training Size: {len(aug_color)} patches")


✅ SR pairs found: 2
✅ Color pairs found: 2

── Final Training Sets ──
🚀 SR Training Size:    16 patches
🚀 Color Training Size: 16 patches


In [114]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — Train SR model
# ═══════════════════════════════════════════════════════════════════
import time, os
import torch, torch.nn as nn
from torch.utils.data import DataLoader, random_split
from models.sr_model import TIRSuperResolutionNet

DEVICE        = torch.device("cuda")
SR_EPOCHS     = 20      
SR_BATCH_SIZE = 8       
SR_LR         = 2e-4

os.makedirs("checkpoints", exist_ok=True)

# We use the dataset we ALREADY built in the Bulletproof cell!
full_ds = aug_sr
print(f"SR patches: {len(full_ds)} effective")

n_val   = max(1, int(len(full_ds) * 0.1))
train_ds, val_ds = random_split(full_ds, [len(full_ds) - n_val, n_val])
train_loader = DataLoader(train_ds, batch_size=SR_BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=SR_BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

model = TIRSuperResolutionNet().to(DEVICE)
opt   = torch.optim.Adam(model.parameters(), lr=SR_LR, betas=(0.9, 0.999))
l1_fn = nn.L1Loss()

print(f"SR params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Epochs: {SR_EPOCHS}")

def psnr_from_mse(mse):
    return 10 * torch.log10(4.0 / (torch.tensor(mse) + 1e-12))

best_val_psnr = -float("inf")
t_start = time.time()

for epoch in range(1, SR_EPOCHS + 1):
    model.train()
    t0, train_loss = time.time(), 0.0
    for low, high in train_loader:
        low, high = low.to(DEVICE), high.to(DEVICE)
        opt.zero_grad()
        loss = l1_fn(model(low), high)
        loss.backward()
        opt.step()
        train_loss += loss.item() * low.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_mse, n = 0.0, 0
    with torch.no_grad():
        for low, high in val_loader:
            low, high = low.to(DEVICE), high.to(DEVICE)
            mse = torch.mean((model(low) - high) ** 2)
            val_mse += mse.item() * low.size(0)
            n += low.size(0)
    val_mse  /= n
    val_psnr  = psnr_from_mse(val_mse).item()
    eta_min   = (time.time() - t_start) / epoch * (SR_EPOCHS - epoch) / 60

    print(f"[SR] {epoch:02d}/{SR_EPOCHS}  L1={train_loss:.4f}  "
          f"val_PSNR={val_psnr:.2f}dB  ({time.time()-t0:.0f}s/ep  ETA={eta_min:.1f}min)")

    torch.save(model.state_dict(), "checkpoints/sr_last.pth")
    if val_psnr > best_val_psnr:
        best_val_psnr = val_psnr
        torch.save(model.state_dict(), "checkpoints/sr_best.pth")
        print("           ✅ Best SR checkpoint saved")

print(f"\n✅ SR done. Best PSNR: {best_val_psnr:.2f} dB  "
      f"({(time.time()-t_start)/60:.1f} min total)")


SR patches: 16 effective
SR params: 785,921
Train: 15  Val: 1  Epochs: 20
[SR] 01/20  L1=0.6686  val_PSNR=7.90dB  (2s/ep  ETA=0.7min)
           ✅ Best SR checkpoint saved
[SR] 02/20  L1=0.5886  val_PSNR=10.18dB  (2s/ep  ETA=0.6min)
           ✅ Best SR checkpoint saved
[SR] 03/20  L1=0.4951  val_PSNR=11.44dB  (2s/ep  ETA=0.6min)
           ✅ Best SR checkpoint saved
[SR] 04/20  L1=0.4899  val_PSNR=12.41dB  (2s/ep  ETA=0.6min)
           ✅ Best SR checkpoint saved
[SR] 05/20  L1=0.4753  val_PSNR=12.44dB  (2s/ep  ETA=0.5min)
           ✅ Best SR checkpoint saved
[SR] 06/20  L1=0.4625  val_PSNR=11.91dB  (2s/ep  ETA=0.5min)
[SR] 07/20  L1=0.4561  val_PSNR=11.50dB  (2s/ep  ETA=0.5min)
[SR] 08/20  L1=0.4524  val_PSNR=11.08dB  (2s/ep  ETA=0.4min)
[SR] 09/20  L1=0.4494  val_PSNR=10.51dB  (2s/ep  ETA=0.4min)
[SR] 10/20  L1=0.4543  val_PSNR=9.78dB  (2s/ep  ETA=0.3min)
[SR] 11/20  L1=0.4465  val_PSNR=9.84dB  (2s/ep  ETA=0.3min)
[SR] 12/20  L1=0.4437  val_PSNR=9.83dB  (2s/ep  ETA=0.3min)
[SR] 13/

In [121]:

# ═══════════════════════════════════════════════════════════════════
# CELL 6 — Train Colorization GAN
# Estimated: ~2.5–3.5 hours for 500 patches × 40 epochs on T4
# ═══════════════════════════════════════════════════════════════════
import time, os
import torch, torch.nn as nn
from torch.utils.data import DataLoader, random_split
from models.colorization_model import UNetGenerator, PatchGANDiscriminator
from models.semantic_constraint import BandRatioConsistencyLoss
from ps10_project.datasets.patch_dataset import ColorizationPatchDataset

from ps10_project.datasets.patch_dataset import ColorizationPatchDataset



DEVICE          = torch.device("cuda")
COLORIZE_EPOCHS = 40      # lower to 25 if tight; 60 is ideal
COLORIZE_BATCH  = 8       # lower to 4 if OOM
LR              = 2e-4
LAMBDA_L1       = 80.0
LAMBDA_SEM      = 5.0

# ── FIX: Pass explicit glob patterns and 3D shapes
base_color = ColorizationPatchDataset(
    "/kaggle/working/ISRO/output/patches",
    tir_glob="*tir*100m*.npy",
    rgb_glob="*rgb*100m*.npy",
    expected_tir_shape=(1, 512, 512)
)
full_ds  = AugmentedColorDataset(base_color)  # 8× augmented
print(f"Color patches: {len(base_color)} real × 8 aug = {len(full_ds)} effective")


n_val   = max(1, int(len(full_ds) * 0.1))
train_ds, val_ds = random_split(full_ds, [len(full_ds) - n_val, n_val])
train_loader = DataLoader(train_ds, batch_size=COLORIZE_BATCH, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=COLORIZE_BATCH, shuffle=False,
                          num_workers=2, pin_memory=True)

G = UNetGenerator().to(DEVICE)
D = PatchGANDiscriminator().to(DEVICE)
opt_g = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
bce         = nn.BCEWithLogitsLoss()
l1_fn       = nn.L1Loss()
semantic_fn = BandRatioConsistencyLoss()

print(f"G params: {sum(p.numel() for p in G.parameters() if p.requires_grad):,}")
print(f"D params: {sum(p.numel() for p in D.parameters() if p.requires_grad):,}")
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Epochs: {COLORIZE_EPOCHS}")

# VRAM headroom check
torch.cuda.empty_cache()
free_gb = torch.cuda.mem_get_info()[0] / 1e9
print(f"Free VRAM: {free_gb:.1f} GB  {'✅ OK' if free_gb > 4 else '⚠️  Consider batch_size=4'}")

best_val_l1 = float("inf")
t_start     = time.time()

for epoch in range(1, COLORIZE_EPOCHS + 1):
    G.train(); D.train()
    t0 = time.time()
    running = {"g": 0.0, "d": 0.0, "l1": 0.0, "sem": 0.0}

    for tir, rgb_real in train_loader:
        tir, rgb_real = tir.to(DEVICE), rgb_real.to(DEVICE)
        bsz = tir.size(0)

        # ── Discriminator ─────────────────────────────────
        with torch.no_grad():
            rgb_fake = G(tir)
        opt_d.zero_grad()
        loss_d = 0.5 * (bce(D(tir, rgb_real), torch.ones_like(D(tir, rgb_real))) +
                        bce(D(tir, rgb_fake.detach()), torch.zeros_like(D(tir, rgb_fake.detach()))))
        loss_d.backward()
        opt_d.step()

        # ── Generator ─────────────────────────────────────
        opt_g.zero_grad()
        rgb_fake      = G(tir)
        d_fake        = D(tir, rgb_fake)
        loss_g_adv    = bce(d_fake, torch.ones_like(d_fake))
        loss_g_l1     = l1_fn(rgb_fake, rgb_real) * LAMBDA_L1
        loss_g_sem    = semantic_fn(tir, rgb_fake) * LAMBDA_SEM
        loss_g        = loss_g_adv + loss_g_l1 + loss_g_sem
        loss_g.backward()
        opt_g.step()

        running["g"]   += loss_g.item() * bsz
        running["d"]   += loss_d.item() * bsz
        running["l1"]  += loss_g_l1.item() * bsz
        running["sem"] += loss_g_sem.item() * bsz

    n = len(train_ds)
    eta_min = (time.time() - t_start) / epoch * (COLORIZE_EPOCHS - epoch) / 60
    print(f"[Color] {epoch:02d}/{COLORIZE_EPOCHS}  "
          f"G={running['g']/n:.3f}  D={running['d']/n:.3f}  "
          f"L1={running['l1']/n:.3f}  Sem={running['sem']/n:.3f}  "
          f"({time.time()-t0:.0f}s/ep  ETA={eta_min:.1f}min)")

    # ── Validation ────────────────────────────────────────
    G.eval()
    val_l1, nval = 0.0, 0
    with torch.no_grad():
        for tir, rgb_real in val_loader:
            tir, rgb_real = tir.to(DEVICE), rgb_real.to(DEVICE)
            val_l1 += l1_fn(G(tir), rgb_real).item() * tir.size(0)
            nval   += tir.size(0)
    val_l1 /= nval
    print(f"           val_L1={val_l1:.4f}")

    torch.save(G.state_dict(), "checkpoints/colorize_G_last.pth")
    torch.save(D.state_dict(), "checkpoints/colorize_D_last.pth")
    if val_l1 < best_val_l1:
        best_val_l1 = val_l1
        torch.save(G.state_dict(), "checkpoints/colorize_G_best.pth")
        print("           ✅ Best colorization checkpoint saved")

print(f"\n✅ Colorization done. Best val L1: {best_val_l1:.4f}  "
      f"({(time.time()-t_start)/60:.1f} min total)")



Color patches: 2 real × 8 aug = 16 effective
G params: 54,402,627
D params: 2,765,377
Train: 15  Val: 1  Epochs: 40
Free VRAM: 12.6 GB  ✅ OK
[Color] 01/40  G=64.024  D=0.642  L1=62.653  Sem=0.155  (3s/ep  ETA=2.0min)
           val_L1=0.5123
           ✅ Best colorization checkpoint saved
[Color] 02/40  G=53.976  D=0.611  L1=52.445  Sem=0.141  (3s/ep  ETA=2.3min)
           val_L1=0.4460
           ✅ Best colorization checkpoint saved
[Color] 03/40  G=46.788  D=0.523  L1=45.406  Sem=0.124  (3s/ep  ETA=2.3min)
           val_L1=0.4008
           ✅ Best colorization checkpoint saved
[Color] 04/40  G=41.709  D=0.494  L1=40.226  Sem=0.114  (3s/ep  ETA=2.3min)
           val_L1=0.3667
           ✅ Best colorization checkpoint saved
[Color] 05/40  G=37.967  D=0.444  L1=36.326  Sem=0.099  (3s/ep  ETA=2.3min)
           val_L1=0.3420
           ✅ Best colorization checkpoint saved
[Color] 06/40  G=34.871  D=0.407  L1=33.152  Sem=0.086  (3s/ep  ETA=2.3min)
           val_L1=0.3229
           ✅ 

In [115]:

# ═══════════════════════════════════════════════════════════════════
# CELL 7 — Run Inference
# Replace INPUT_TIF with your real 200m GeoTIFF path.
# ═══════════════════════════════════════════════════════════════════
import torch, os, tifffile
import torch.nn.functional as F
import numpy as np
from models.sr_model import TIRSuperResolutionNet
from models.colorization_model import UNetGenerator

DEVICE = torch.device("cuda")

sr_model = TIRSuperResolutionNet().to(DEVICE)
sr_model.load_state_dict(torch.load("checkpoints/sr_best.pth", map_location=DEVICE))
sr_model.eval()

colorize_model = UNetGenerator().to(DEVICE)
colorize_model.load_state_dict(torch.load("checkpoints/colorize_G_best.pth", map_location=DEVICE))
colorize_model.eval()
print("✅ Models loaded on GPU")

# ── Set INPUT_TIF to your real scene, or leave None for synthetic test ──
INPUT_TIF = None  # e.g. "/kaggle/working/raw_tiles/LC09_B10_200m.tif"

if INPUT_TIF is None:
    fake = np.random.uniform(0, 1, (256, 256)).astype(np.float32)
    os.makedirs("/kaggle/working/fake_input", exist_ok=True)
    tifffile.imwrite("/kaggle/working/fake_input/TEST_B10_200m.tif", fake)
    INPUT_TIF = "/kaggle/working/fake_input/TEST_B10_200m.tif"
    print("⚠️  Using synthetic input. Set INPUT_TIF to your real .tif for submission.")

from inference import run_inference
sr_path, color_path = run_inference(
    input_tir_path=INPUT_TIF,
    sr_model=sr_model,
    colorize_model=colorize_model,
    device=DEVICE,
    product_id=os.path.splitext(os.path.basename(INPUT_TIF))[0],
    output_root="/kaggle/working/output/model_outputs"
)
print("✅ Inference done")
print("SR output:        ", sr_path)
print("Colorized output: ", color_path)

arr = tifffile.imread(color_path)
print(f"Output shape: {arr.shape}  dtype: {arr.dtype}  "
      f"(channels = Blue, Green, Red per ISRO spec)")



✅ Models loaded on GPU
⚠️  Using synthetic input. Set INPUT_TIF to your real .tif for submission.
✅ Inference done
SR output:         /kaggle/working/output/model_outputs/tir_superresolved_100m/TEST_B10_200m.tif
Colorized output:  /kaggle/working/output/model_outputs/colorized_tir_100m/TEST_B10_200m.tif
Output shape: (3, 256, 256)  dtype: uint8  (channels = Blue, Green, Red per ISRO spec)


/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
/usr/local/lib/python3.12/dist-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(


In [28]:
import shutil, os

WORK_DIR = "/kaggle/working/ps10_project"   # redefine in case kernel lost it

# Submission outputs
SUBMIT_ZIP = "/kaggle/working/ps10_model_outputs.zip"
shutil.make_archive(SUBMIT_ZIP.replace(".zip",""), "zip",
                    root_dir=WORK_DIR, base_dir="../output/model_outputs")
print(f"✅ Outputs zip: {SUBMIT_ZIP}  ({os.path.getsize(SUBMIT_ZIP)/1e6:.1f} MB)")

# Checkpoints — note root_dir is WORK_DIR now, not /kaggle/working
CKPT_ZIP = "/kaggle/working/ps10_checkpoints.zip"
ckpt_path = os.path.join(WORK_DIR, "checkpoints")
if os.path.exists(ckpt_path):
    shutil.make_archive(CKPT_ZIP.replace(".zip",""), "zip",
                        root_dir=WORK_DIR, base_dir="checkpoints")
    print(f"✅ Checkpoints: {CKPT_ZIP}  ({os.path.getsize(CKPT_ZIP)/1e6:.1f} MB)")
else:
    print(f"❌ Checkpoints dir not found at {ckpt_path}")
    print("   List WORK_DIR:", os.listdir(WORK_DIR))

print("\n→ Download from the Output panel (right side of Kaggle)")


✅ Outputs zip: /kaggle/working/ps10_model_outputs.zip  (0.2 MB)
✅ Checkpoints: /kaggle/working/ps10_checkpoints.zip  (419.2 MB)

→ Download from the Output panel (right side of Kaggle)


In [126]:
import os
import torch
import numpy as np
import tifffile
from models.sr_model import TIRSuperResolutionNet
from models.colorization_model import UNetGenerator

DEVICE = torch.device("cpu")

# ── 1. Load Trained Models (Using exact absolute paths) ──
sr_model = TIRSuperResolutionNet().to(DEVICE)
sr_model.load_state_dict(torch.load("/kaggle/working/ISRO/checkpoints/sr_best.pth", map_location=DEVICE))
sr_model.eval()

colorize_model = UNetGenerator(in_channels=1, out_channels=3).to(DEVICE)
colorize_model.load_state_dict(torch.load("/kaggle/working/ISRO/checkpoints/colorize_G_best.pth", map_location=DEVICE))
colorize_model.eval()

# ── 2. Setup output directories ──
out_sr = "/kaggle/working/output/model_outputs/tir_superresolved_100m"
out_color = "/kaggle/working/output/model_outputs/colorized_tir_100m"
os.makedirs(out_sr, exist_ok=True)
os.makedirs(out_color, exist_ok=True)

# ── 3. Find your real input TIF ──
import glob
real_tifs = glob.glob("/kaggle/working/ISRO/input/**/*B10*.tif", recursive=True) + \
            glob.glob("/kaggle/working/raw_tiles/**/*B10*.tif", recursive=True)

if not real_tifs:
    print("❌ Could not find the raw B10 TIF file to run inference on! Upload your real TIF to Kaggle.")
else:
    real_input = real_tifs[0]
    product_id = os.path.splitext(os.path.basename(real_input))[0]
    print(f"🚀 Running Inference on real data: {real_input}")
    
    # Load Real Data
    try:
        import rasterio
        with rasterio.open(real_input) as src:
            tir = src.read(1)
            profile = src.profile.copy()
            # To be a correct 100m map, we halve the 200m pixel size in the transform
            transform = profile['transform'] * rasterio.Affine.scale(0.5, 0.5)
    except ImportError:
        tir = tifffile.imread(real_input)
        if tir.ndim == 3: tir = tir[..., 0]
        profile = None

    # Normalize
    tir = tir.astype(np.float32)
    lo, hi = np.percentile(tir, 1), np.percentile(tir, 99)
    if hi - lo < 1e-6: hi = lo + 1e-6
    tir_tanh = np.clip((tir - lo) / (hi - lo), 0, 1) * 2 - 1

    # ── FIX: Pad input to a multiple of 256 so the 9-level U-Net doesn't get shape mismatches ──
    H, W = tir_tanh.shape
    pad_h = (256 - H % 256) % 256
    pad_w = (256 - W % 256) % 256
    tir_padded = np.pad(tir_tanh, ((0, pad_h), (0, pad_w)), mode='reflect')

    # Run Models
    with torch.no_grad():
        x = torch.from_numpy(tir_padded).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
        
        # 1. Super Resolution
        sr_out = sr_model(x) 
        sr_np = sr_out.squeeze().cpu().numpy()
        
        # 2. Colorization
        rgb_out = colorize_model(sr_out) 
        rgb_np = rgb_out.squeeze().cpu().numpy() 

    # ── FIX: Crop back to the exact target size (2x the original input size) ──
    H_out, W_out = H * 2, W * 2
    sr_np = sr_np[:H_out, :W_out]
    rgb_np = rgb_np[:, :H_out, :W_out]

    # Denormalize
    sr_uint8 = np.clip(((sr_np + 1) / 2) * 255.0, 0, 255).astype(np.uint8)
    rgb_uint8 = np.clip(((rgb_np + 1) / 2) * 255.0, 0, 255).astype(np.uint8)
    
    # Mandatory Band Order: Blue, Green, Red
    bgr_stack = np.stack([rgb_uint8[2], rgb_uint8[1], rgb_uint8[0]], axis=0)

    # Save
    sr_path = os.path.join(out_sr, f"{product_id}.tif")
    color_path = os.path.join(out_color, f"{product_id}.tif")
    
    if profile:
        profile.update(dtype="uint8", count=1, height=H_out, width=W_out, transform=transform, driver="GTiff")
        with rasterio.open(sr_path, "w", **profile) as dst:
            dst.write(sr_uint8, 1)
            
        profile.update(count=3)
        with rasterio.open(color_path, "w", **profile) as dst:
            dst.write(bgr_stack)
            dst.set_band_description(1, "Blue")
            dst.set_band_description(2, "Green")
            dst.set_band_description(3, "Red")
    else:
        tifffile.imwrite(sr_path, sr_uint8)
        tifffile.imwrite(color_path, np.moveaxis(bgr_stack, 0, -1))
        
    print(f"✅ Final SR TIF saved: {sr_path} (Shape: {sr_uint8.shape})")
    print(f"✅ Final Color TIF saved: {color_path} (Shape: {bgr_stack.shape})")

# ── 4. Zip EVERYTHING for Submission ──
import subprocess
print("\n📦 Zipping up your submission...")
subprocess.run("zip -r /kaggle/working/submission.zip /kaggle/working/output/model_outputs /kaggle/working/ISRO/checkpoints", shell=True)
print("🎉 All done! Download 'submission.zip' from the Kaggle Output sidebar.")


🚀 Running Inference on real data: /kaggle/working/ISRO/input/tile_SW/demo_B10.tif
✅ Final SR TIF saved: /kaggle/working/output/model_outputs/tir_superresolved_100m/demo_B10.tif (Shape: (5618, 5010))
✅ Final Color TIF saved: /kaggle/working/output/model_outputs/colorized_tir_100m/demo_B10.tif (Shape: (3, 5618, 5010))

📦 Zipping up your submission...
  adding: kaggle/working/output/model_outputs/ (stored 0%)
  adding: kaggle/working/output/model_outputs/tir_superresolved_100m/ (stored 0%)
  adding: kaggle/working/output/model_outputs/tir_superresolved_100m/TEST_B10_200m.tif (deflated 42%)
  adding: kaggle/working/output/model_outputs/tir_superresolved_100m/demo_B10.tif (deflated 3%)
  adding: kaggle/working/output/model_outputs/colorized_tir_100m/ (stored 0%)
  adding: kaggle/working/output/model_outputs/colorized_tir_100m/TEST_B10_200m.tif (deflated 22%)
  adding: kaggle/working/output/model_outputs/colorized_tir_100m/demo_B10.tif (deflated 3%)
  adding: kaggle/working/ISRO/checkpoints/